In [4]:
import os
import cv2
from PIL import Image
from tqdm.notebook import tqdm
import numpy as np
gt_path = '/home/shenss/python/dataset/Diagnosis of Diabetic Retinopathy/test/No_DR'
fs_path = '/home/shenss/python/dataset/Diagnosis of Diabetic Retinopathy/test/No_DR_FS'
os.makedirs(fs_path, exist_ok=True)

In [ ]:
for img_name in tqdm(os.listdir(gt_path)):
    img_path = os.path.join(gt_path, img_name)
    img = Image.open(img_path).resize((256, 256), Image.BICUBIC)
    img.save(os.path.join(gt_path, img_name), format='png')

  0%|          | 0/109 [00:00<?, ?it/s]

In [5]:
def ht_color(img):
    (r, g, b) = img[:, :, 0], img[:, :, 1], img[:, :, 2]
    r = Image.fromarray(r).convert('1').convert('L')
    g = Image.fromarray(g).convert('1').convert('L')
    b = Image.fromarray(b).convert('1').convert('L')
    return Image.merge('RGB', (r, g, b))

for img_name in tqdm(os.listdir(gt_path)):
    img_path = os.path.join(gt_path, img_name)
    img = Image.open(img_path)
    img = ht_color(np.array(img))
    img = np.array(img)
    img = cv2.GaussianBlur(img, (3, 3), 0)
    img = Image.fromarray(img)

    img.save(os.path.join(fs_path, img_name), format='png')

  0%|          | 0/118 [00:00<?, ?it/s]

In [5]:
import os
import cv2
from PIL import Image
from tqdm.notebook import tqdm
import numpy as np
gt_path = '/home/shenss/python/dataset/VOC2012_ORI/train/ht_Gaussian'
sobel_path = '/home/shenss/python/dataset/VOC2012_ORI/train/GSobel'
os.makedirs(sobel_path, exist_ok=True)

In [6]:
def weighted_sobel(img):
    img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    img_sobel_x = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    img_sobel_x = cv2.convertScaleAbs(img_sobel_x)
    img_sobel_y = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    img_sobel_y = cv2.convertScaleAbs(img_sobel_y)
    img_sobel = cv2.addWeighted(img_sobel_x, 0.5, img_sobel_y, 0.5, 0)
    return img_sobel

for img_name in tqdm(os.listdir(gt_path)):
    img_path = os.path.join(gt_path, img_name)
    img = cv2.imread(img_path)
    img = weighted_sobel(img)
    cv2.imwrite(os.path.join(sobel_path, img_name), img)


  0%|          | 0/13469 [00:00<?, ?it/s]

In [7]:
import os
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

def process_image(img, gs=False):
    """
    对输入图像应用HSV转换和二值化处理
    
    参数:
        img: 输入的BGR图像
    
    返回:
        处理后的BGR图像
    """
    # 转换为HSV空间
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    # 将V通道值减少到原来的1/4
    img_hsv[:, :, 2] = (img_hsv[:, :, 2] / 4).astype(np.uint8)
    
    # 转回BGR空间
    img_bgr = cv2.cvtColor(img_hsv, cv2.COLOR_HSV2BGR)
    
    # 分离通道
    (b, g, r) = cv2.split(img_bgr)
    
    # 对每个通道进行二值化处理
    b = np.array(Image.fromarray(b).convert("1").convert("L"))
    g = np.array(Image.fromarray(g).convert("1").convert("L"))
    r = np.array(Image.fromarray(r).convert("1").convert("L"))
    
    # 合并通道
    processed_img = cv2.merge((b, g, r))

    if gs:
        # 如果需要，应用高斯模糊
        processed_img = cv2.GaussianBlur(processed_img, (3, 3), 0)

    processed_img = Image.fromarray(cv2.cvtColor(processed_img, cv2.COLOR_BGR2RGB))
    
    return processed_img

def main():
    # 输入和输出目录
    input_dir = "/home/shenss/python/dataset/Diagnosis of Diabetic Retinopathy/test/DR"
    output_dir = "/home/shenss/python/dataset/Diagnosis of Diabetic Retinopathy/test/DR_EVCS"
    
    # 创建输出目录
    os.makedirs(os.path.join(output_dir), exist_ok=True)
    
    # 获取目录中的所有图像文件
    image_files = [f for f in os.listdir(input_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    image_files.sort()  # 确保排序，以便始终获取相同的"前20张"
    
    # 只处理前20张图像
    # image_files = image_files[:20]
    
    print(f"将处理 {len(image_files)} 张图像...")
    
    # 处理每张图像
    for i, filename in enumerate(tqdm(image_files)):
        # 读取图像
        img_path = os.path.join(input_dir, filename)
        img = cv2.imread(img_path)
        
        if img is None:
            print(f"无法读取图像 {img_path}，跳过")
            continue
        
        # 保存原图
        # original_output_path = os.path.join(output_dir, "original", filename)
        # cv2.imwrite(original_output_path, img)
        
        # 处理图像
        processed_img = process_image(img, gs=True)
        
        # 保存处理后的图像
        processed_output_path = os.path.join(output_dir, filename)
        processed_img.save(processed_output_path, format='png')
        # png 格式
        
        
    print("处理完成！")
    

if __name__ == "__main__":
    main()

将处理 113 张图像...


100%|██████████| 113/113 [00:01<00:00, 79.18it/s]

处理完成！
